# 07 · Una cabeza nueva y un adaptador que aprende

¿Qué aprende un adaptador si casi todo el modelo está congelado? Este
experimento permite verlo: el encoder es un **BERT diminuto inicializado al
azar**, y la tarea distingue dos grupos de tokens sintéticos. No reconoce
lenguaje y no usa conocimiento preentrenado. Sirve para explorar la mecánica
de PEFT antes de aplicarla a un modelo permitido por una tarea real.

Esta referencia entrena adaptadores y cabeza.
La configuración conserva la cabeza nueva como entrenable. Prueba qué ocurre
al omitir `modules_to_save` y mira sus parámetros, sin confundir un código
que corre con una cabeza que aprende.

Todo usa CPU, sin red ni archivos de modelos externos. Datos sintéticos
generados con semilla fija, CC0-1.0. Requiere el entorno de estudio con
PyTorch, Transformers, PEFT y matplotlib. 60–90 minutos de exploración, sin límite de juez.


In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertConfig, BertModel
import matplotlib.pyplot as plt

torch.set_num_threads(1)
torch.manual_seed(4)
g = torch.Generator().manual_seed(41)
def muestras(n, longitud):
    y = torch.arange(n) % 2
    ids = torch.randint(1, 24, (n, longitud), generator=g) + y[:, None] * 24
    return ids, torch.ones_like(ids), y
train = muestras(96, 5)
val = muestras(48, 5)
nuevo = muestras(48, 10)
encoder = BertModel(BertConfig(vocab_size=48, hidden_size=32,
    intermediate_size=64, num_hidden_layers=1, num_attention_heads=4,
    hidden_dropout_prob=0., attention_probs_dropout_prob=0.))
pesos_base = [(p, p.detach().clone()) for p in encoder.parameters()]


In [ ]:
# bloque: probado; id: clasificador_texto
import torch

class Clasificador(torch.nn.Module):
    def __init__(self, encoder, n_clases):
        super().__init__()
        self.encoder = encoder
        self.cabeza = torch.nn.Linear(encoder.config.hidden_size, n_clases)

    def forward(self, ids, mascara):
        h = self.encoder(input_ids=ids, attention_mask=mascara).last_hidden_state
        m = mascara.unsqueeze(-1)
        vector = (h * m).sum(1) / m.sum(1).clamp(min=1)     # mean pooling enmascarado
        return self.cabeza(vector)                          # logits, sin softmax


In [ ]:
modelo = Clasificador(encoder, n_clases=2).eval()
with torch.no_grad():
    logits_sin_lora = modelo(val[0], val[1]).clone()


## La actualización empieza en cero; sus factores no

En la inicialización estándar, A es aleatoria y B es cero. El producto
`B @ A` es cero, de modo que insertar el adaptador no cambia inicialmente
la función. Si los dos factores fueran cero, ambos gradientes serían cero.
La cabeza es nueva y también tiene que aprender: PEFT la conserva con
`modules_to_save=["cabeza"]`.


In [ ]:
# bloque: probado; id: lora_cabeza
from peft import LoraConfig, get_peft_model

config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                    target_modules=["query", "value"],
                    modules_to_save=["cabeza"])  # la cabeza NUEVA también aprende
modelo = get_peft_model(modelo, config)
modelo.print_trainable_parameters()      # la proporción depende de la arquitectura


In [ ]:
modelo.eval()
with torch.no_grad():
    logits_iniciales = modelo(val[0], val[1])
assert torch.allclose(logits_iniciales, logits_sin_lora, atol=1e-6)
entrenables = {n: p for n, p in modelo.named_parameters() if p.requires_grad}
antes = {n: p.detach().clone() for n, p in entrenables.items()}
for nombre, p in entrenables.items():
    print(nombre, tuple(p.shape))
print("Entrenables:", sum(p.numel() for p in entrenables.values()))
print("Totales:", sum(p.numel() for p in modelo.parameters()))


In [ ]:
EPOCAS = 20
LR = 0.01  # tarea sintética diminuta; no trasladar este valor a un encoder grande


In [ ]:
def medir(datos):
    modelo.eval()
    with torch.no_grad():
        logits = modelo(datos[0], datos[1])
        perdida = nn.functional.cross_entropy(logits, datos[2]).item()
        accuracy = (logits.argmax(1) == datos[2]).float().mean().item()
    return perdida, accuracy

baseline = medir(val)[1]
optimizador = torch.optim.AdamW(entrenables.values(), lr=LR, weight_decay=0.)
cargador = DataLoader(TensorDataset(*train), batch_size=24, shuffle=True,
                     generator=torch.Generator().manual_seed(8))
historial = [(*medir(train), *medir(val))]
for epoca in range(EPOCAS):
    modelo.train()
    for ids, mascara, y in cargador:
        optimizador.zero_grad(set_to_none=True)
        perdida = nn.functional.cross_entropy(modelo(ids, mascara), y)
        perdida.backward()
        optimizador.step()
    historial.append((*medir(train), *medir(val)))
resultado = {"baseline": baseline, "validacion": medir(val)[1],
             "transferencia": medir(nuevo)[1]}
curvas = np.asarray(historial)
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
ejes[0].plot(curvas[:, 0], label="train")
ejes[0].plot(curvas[:, 2], label="validación")
ejes[0].set(xlabel="época", ylabel="cross-entropy"); ejes[0].legend()
ejes[1].plot(curvas[:, 1], label="train")
ejes[1].plot(curvas[:, 3], label="validación")
ejes[1].set(xlabel="época", ylabel="accuracy", ylim=(0, 1.05)); ejes[1].legend()
fig.tight_layout(); plt.show()


## Ver la matriz que aprendió

Cada mapa tiene 32 filas y 32 columnas: una actualización de la proyección
query. Aunque el adaptador la representa mediante dos factores, su efecto
se suma a la matriz completa. El factor `alpha/r` también importa.
Una actualización distinta de cero no implica que mejore cualquier tarea.


In [ ]:
adaptador = next(m for m in modelo.modules() if hasattr(m, "lora_A") and "default" in m.lora_A)
A = adaptador.lora_A["default"].weight.detach()
B = adaptador.lora_B["default"].weight.detach()
delta = (B @ A * adaptador.scaling["default"]).cpu().numpy()
figura, eje = plt.subplots(figsize=(4.5, 4))
limite = max(float(np.abs(delta).max()), 1e-8)
mapa = eje.imshow(delta, cmap="RdBu_r", vmin=-limite, vmax=limite)
eje.set(title="Actualización LoRA de query", xlabel="entrada", ylabel="salida")
figura.colorbar(mapa, ax=eje); figura.tight_layout(); plt.show()
cambios = {n: float((p.detach() - antes[n]).abs().max()) for n, p in entrenables.items()}
resultado["cambio_maximo_cabeza"] = max(v for n, v in cambios.items() if "cabeza" in n)
resultado["cambio_maximo_adaptador"] = max(v for n, v in cambios.items() if "lora_" in n)
resultado["encoder_base_intacto"] = all(torch.equal(p, original) for p, original in pesos_base)


## Cambiar la pregunta

La transferencia duplica la longitud y conserva la regla de tokens. Compara
ambas accuracies y luego mezcla algunos tokens del otro grupo: el problema
ya no tiene la misma señal.

Mira los nombres de los parámetros de la cabeza con y sin `modules_to_save`.
Un porcentaje pequeño de parámetros entrenables no demuestra que hayas
elegido los correctos. Después prueba r=2 y r=16 y observa tamaño de la
actualización, tiempo y validación.

Para pasar a lenguaje real necesitas una preparación separada con el encoder
permitido, su tokenizador, datos de train y validación apropiados. Este
notebook no mide el beneficio del preentrenamiento ni valida una ejecución
oficial o en GPU. El learning rate del juguete tampoco calibra ese caso.


In [ ]:
assert resultado["validacion"] > resultado["baseline"]
assert resultado["validacion"] >= 0.85 and resultado["transferencia"] >= 0.85
assert resultado["cambio_maximo_cabeza"] > 0
assert resultado["cambio_maximo_adaptador"] > 0
assert resultado["encoder_base_intacto"]


In [ ]:
resultado.update({"laboratorio": '07_lora',
                  "version": 'solución de referencia',
                  "metrica": 'accuracy sobre secuencias de tokens sintéticos (mayor es mejor)', "split": '96 train, 48 validación y 48 transferencia; generador torch con semilla 41, transferencia duplica longitud de 5 a 10'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False,
                  indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
